In [ ]:
import pandas as pd
from rdkit import Chem

# 1. 读取 CSV 文件
df = pd.read_csv('./SOS1/glide-dock_SP_1_SOS1_all_molecules_by_pocket_similarity.csv')

# 2. 定义计算重原子数的函数
def get_heavy_atom_count(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return mol.GetNumHeavyAtoms()
    return None

# 3. 计算每个分子的重原子数
df['Heavy_Atoms'] = df['Original_SMILES'].apply(get_heavy_atom_count)

# 4. 计算配体效率 (Docking Score / 重原子数)
df['Ligand_Efficiency'] = df['docking score'] / df['Heavy_Atoms']

# 5. 查看前几行结果并保存
print(df.head())
df.to_csv('./SOS1/glide-dock_SP_1_SOS1_all_molecules_by_pocket_similarity_ligand_efficiency_results.csv', index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import os

# 1. 文件与颜色映射
file_mapping = {
    "compared_pocket_sim": {
        "file": "./SOS1/XP_1_compared_pocket_similarity_ligand_efficiency_results.csv",
        "color": "#2B72D2",
        "label": "compared_pocket_sim"
    },
    "key_aas_sim": {
        "file": "./SOS1/XP_1_keyaas_similarity_ligand_efficiency_results.csv",
        "color": "#7F57BF",
        "label": "key_aas_sim"
    },
    "pocket_seq_sim": {
        "file": "./SOS1/XP_1_pocket_seq_similarity_ligand_efficiency_results.csv",
        "color": "#D456AE",
        "label": "pocket_seq_sim"
    },
    "pocket_sim": {
        "file": "./SOS1/XP_1_pocket_similarity_ligand_efficiency_results.csv",
        "color": "#FF668A",
        "label": "pocket_sim"
    },
    "decorator": {
        "file": "./SOS1/XP_1_decorator_similarity_ligand_efficiency_results.csv",
        "color": "#FF9670",
        "label": "decorator"
    },
    "asinex": {
        "file": "./SOS1/XP_1_random_asinex_ligand_efficiency_results.csv",
        "color": "#FFC85F",
        "label": "asinex"
    },
    "chemdiv": {
        "file": "./SOS1/XP_1_random_chemdiv_ligand_efficiency_results.csv",
        "color": "#F9F872",
        "label": "chemdiv"
    }
}

# 2. 读取数据、处理 Title 列并收集 Ligand Efficiency
le_data_dict = {}

for name, info in file_mapping.items():
    file_path = info["file"]

    if not os.path.exists(file_path):
        print(f"Warning: 找不到文件 {file_path}，已跳过。")
        continue

    df = pd.read_csv(file_path, low_memory=False)

    # 处理 Title 列：以 '_' 分割，取最后一部分作为 Label
    if "Title" in df.columns:
        df["Label"] = df["Title"].astype(str).apply(lambda x: x.split('_')[-1] if '_' in x else x)
    else:
        print(f"Warning: {name} 中未找到 'Title' 列。")

    # 保存为新的 CSV 文件
    save_name = file_path.replace(".csv", "_labeled.csv")
    # df.to_csv(save_name, index=False)
    print(f"已处理并保存: {save_name}")

    # 收集 Ligand_Efficiency 列用于绘图
    if "Ligand_Efficiency" in df.columns:
        le_values = df["Ligand_Efficiency"].dropna()
        le_data_dict[name] = le_values
    else:
        print(f"Warning: {name} 中未找到 'Ligand_Efficiency' 列。")

# 3. 全局绘图设置：全白背景 + 无网格
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 300
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.grid'] = False

# 4. 绘制配体效率核密度分布图 (KDE)
if le_data_dict:
    plt.figure(figsize=(16, 8))

    # 使用 Seaborn 绘制每组数据的 KDE 曲线
    for name, le_values in le_data_dict.items():
        sns.kdeplot(
            data=le_values,
            color=file_mapping[name]["color"],
            label=file_mapping[name]["label"],
            linewidth=2.5,
            fill=True,       # 曲线下方添加半透明填充
            alpha=0.15       # 填充的透明度
        )

    # 坐标轴与标题配置
    plt.yticks(fontsize=15, fontweight='bold')
    plt.xticks(fontsize=15, fontweight='bold')

    plt.ylabel("Density", fontsize=16, fontweight='bold', labelpad=10)
    plt.xlabel("Ligand Efficiency", fontsize=16, fontweight='bold', labelpad=10)
    plt.title("Ligand Efficiency Density Distribution", fontsize=20, fontweight='bold', pad=20)

    # 图例配置
    plt.legend(
        frameon=False,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.15),
        ncol=4,
        prop={"weight": "bold", "size": 15}
    )

    # 移除顶部和右侧的边框线让图表更清爽
    sns.despine()

    plt.tight_layout()

    # 保存图片
    output_img = "./SOS1/ligand_efficiency_density_plot_XP.png"
    plt.savefig(output_img, dpi=300, bbox_inches="tight", facecolor='white')
    plt.show()
    print(f"\n密度分布图已保存为 {output_img}")
else:
    print("没有找到有效的 Ligand_Efficiency 数据进行绘图。")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, roc_auc_score
from rdkit.ML.Scoring import Scoring

def calculate_ef(y_true, y_scores, percentage=0.01):
    """
    计算特定百分比下的富集因子 (Enrichment Factor, EF)
    """
    n_total = len(y_true)
    n_top = max(1, int(n_total * percentage))
    n_actives = sum(y_true)

    if n_actives == 0:
        return 0.0

    # 根据分数从大到小排序的索引
    sorted_indices = np.argsort(y_scores)[::-1]
    y_true_sorted = np.array(y_true)[sorted_indices]

    # 计算前 X% 中的活性分子数
    actives_in_top = sum(y_true_sorted[:n_top])

    # 计算 EF
    ef = (actives_in_top / n_top) / (n_actives / n_total)
    return ef

def calculate_bedroc(y_true, y_scores, alpha=20.0):
    """
    利用 RDKit 计算 BEDROC
    alpha=20.0 代表我们关注前 8% 的早期富集 (80%的权重分布在这一区间)
    alpha=80.5 代表我们关注前 2% 的早期富集
    """
    # RDKit 要求输入格式为列表，每个元素是 (score, label)，并且需要按 score 降序排列
    data = list(zip(y_scores, y_true))
    data.sort(key=lambda x: x[0], reverse=True)

    # 1 代表 label 在 tuple 中的索引，即 x[1]
    bedroc = Scoring.CalcBEDROC(data, 1, alpha)
    return bedroc

# 1. 定义要处理的文件列表和用于图例的短名称
files_info = {
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_compared_pocket_similarity_ligand_efficiency_results_labeled.csv": "compared_pocket_sim",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_keyaas_overlap_ligand_efficiency_results_labeled.csv": "key_aas_smi",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_seq_similarity_ligand_efficiency_results_labeled.csv": "pocket_seq_ sim",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled.csv": "pocket_sim"
}

# 用于保存评估结果
results = []

# 初始化绘图
plt.figure(figsize=(8, 8))
colors = ['#2B72D2', '#7F57BF', '#D456AE', '#FF668A'] # 分配颜色

# 2. 遍历读取文件并计算指标
for (file_path, short_name), color in zip(files_info.items(), colors):
    try:
        # 读取数据
        df = pd.read_csv(file_path)

        # 提取真实标签 (Label) 和 预测分数
        # 假设对接打分越小（负值越大）越好，因此取相反数使其变为“越大越好”
        y_true = df['Label'].values
        y_scores = -df['docking score'].values

        # 计算各种评价指标
        roc_auc = roc_auc_score(y_true, y_scores)
        ef_1 = calculate_ef(y_true, y_scores, percentage=0.01)
        ef_5 = calculate_ef(y_true, y_scores, percentage=0.05)
        bedroc_20 = calculate_bedroc(y_true, y_scores, alpha=20.0)

        # 将结果加入列表
        results.append({
            "Method": short_name,
            "ROC-AUC": roc_auc,
            "EF 1%": ef_1,
            "EF 5%": ef_5,
            "BEDROC (α=20)": bedroc_20
        })

        # 绘制 ROC 曲线
        fpr, tpr, _ = roc_curve(y_true, y_scores)
        plt.plot(fpr, tpr, color=color, lw=2, label=f'{short_name} (AUC = {roc_auc:.3f})')

    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {e}")

# 3. 打印结果比较表
results_df = pd.DataFrame(results)
print("\n" + "="*60)
print("🎯 定量富集评价指标总结 (Virtual Screening Performance)")
print("="*60)
print(results_df.to_string(index=False))
print("="*60 + "\n")

# 4. 完善和保存 ROC 曲线图
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random (AUC = 0.500)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (FPR)', fontsize=12)
plt.ylabel('True Positive Rate (TPR)', fontsize=12)
plt.title('ROC Curves of Different Pocket Selection Strategies', fontsize=14)
plt.legend(loc="lower right", fontsize=10)
plt.grid(alpha=0.3)

# 保存并在屏幕上显示图片
plt.savefig('combined_roc_curves.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 1. 文件与颜色映射
file_mapping = {
    "compared_pocket_sim": "./MK14/glide-dock_SP_1_MK14_all_molecules_by_compared_pocket_similarity.csv",
    "key_aas_sim": "./MK14/glide-dock_SP_1_MK14_all_molecules_by_keyaas_overlap.csv",
    "pocket_seq_sim": "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_seq_similarity.csv",
    "pocket_sim": "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_similarity.csv",
}

custom_color_dict = {
    "compared_pocket_sim": "#2B72D2",
    "key_aas_sim": "#7F57BF",
    "pocket_seq_sim": "#D456AE",
    "pocket_sim": "#FF668A",
}

# 2. 分箱区间
bins = np.arange(-12, 2.1, 1)
bin_labels = [f"[{i}.0, {i+1}.0)" for i in bins[:-1]]

# 3. 读取数据统计分箱
counts_dict = {}
for name, file_path in file_mapping.items():
    df = pd.read_csv(file_path, low_memory=False)
    docking_scores = df["docking score"].dropna()
    counts = pd.cut(
        docking_scores,
        bins=bins,
        labels=bin_labels,
        right=False
    ).value_counts().sort_index()
    counts_dict[name] = counts

# 4. 全局绘图设置：全白背景 + 无网格
plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 12
plt.rcParams["figure.dpi"] = 300
plt.rcParams['axes.labelweight'] = 'bold'
plt.rcParams['axes.facecolor'] = 'white'       # 坐标轴背景纯白
plt.rcParams['figure.facecolor'] = 'white'     # 画布背景纯白
plt.rcParams['axes.grid'] = False              # 关闭网格线

plt.figure(figsize=(18, 9))

x = np.arange(len(bin_labels))
bar_width = 0.10   # 从 0.15 减小到 0.10，7 根总宽 0.7，留出间距

# 绘制分组柱状图，offset 居中对齐
n_files = len(counts_dict)
for idx, (name, counts) in enumerate(counts_dict.items()):
    offset = bar_width * (idx - (n_files - 1) / 2)
    plt.bar(
        x + offset,
        counts,
        width=bar_width,
        label=name,
        color=custom_color_dict[name],
        edgecolor='white',   # 白色描边，进一步区分相邻柱子
        linewidth=0.5
    )

# 5. 坐标轴配置
plt.xticks(x, bin_labels, rotation=45, ha='right', fontweight='bold')
plt.yticks(fontsize=15, fontweight='bold')

plt.ylabel("Count", fontsize=16, fontweight='bold', labelpad=10)
plt.title("Docking Score Distribution", fontsize=20, fontweight='bold', pad=20)

plt.legend(
    frameon=False,
    loc="upper center",
    bbox_to_anchor=(0.5, -0.18),
    ncol=5,
    fontsize=15,
    prop={"weight": "bold"}
)

plt.ylim(0, 6000)
plt.tight_layout()

plt.savefig("./MK14/docking_score_histogram_MK14.png", dpi=300, bbox_inches="tight",
            facecolor='white')   # 保存时也强制白底
plt.show()

In [ ]:
import pandas as pd

# 读取指定文件
file_name = "./KIT/glide-dock_SP_1_KIT_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled.csv"
df = pd.read_csv(file_name)

# 按照 'docking score' 升序排序，确保得分最小的行在最前
df_sorted = df.sort_values(by='docking score', ascending=True)

# 根据 'Title' 列去重，keep='first' 会保留得分最小的那一行
df_filtered = df_sorted.drop_duplicates(subset=['Title'], keep='first')

# 将结果保存为新的 CSV 文件
output_file = "./KIT/glide-dock_SP_1_KIT_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled_dep.csv"
df_filtered.to_csv(output_file, index=False)

print(f"数据处理完成：从 {len(df)} 行过滤至 {len(df_filtered)} 行。")

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_auc_score
from rdkit.ML.Scoring import Scoring

def calculate_ef(y_true, y_scores, percentage):
    """
    计算特定百分比下的富集因子 (Enrichment Factor, EF)
    """
    n_total = len(y_true)
    n_top = max(1, int(n_total * percentage))
    n_actives = sum(y_true)

    if n_actives == 0:
        return 0.0

    # 此时 y_scores 已经是按从大到小（对接打分从小到大）对齐的
    # 保险起见，依然通过 argsort 获取最高分的索引
    sorted_indices = np.argsort(y_scores)[::-1]
    y_true_sorted = np.array(y_true)[sorted_indices]
    actives_in_top = sum(y_true_sorted[:n_top])

    ef = (actives_in_top / n_top) / (n_actives / n_total)
    return ef

def calculate_bedroc_at_percentage(y_true, y_scores, percentage):
    """
    计算特定百分比下的 BEDROC
    根据公式 alpha = -ln(0.2) / percentage，将 80% 的评价权重集中在前 percentage 的分子上
    """
    alpha = 1.609438 / percentage
    data = list(zip(y_scores, y_true))
    # RDKit 要求输入数据按分数降序排列
    data.sort(key=lambda x: x[0], reverse=True)

    try:
        # 当 alpha 大于 ~709 时，RDKit 底层可能会因为 e^alpha 溢出抛出异常
        return Scoring.CalcBEDROC(data, 1, alpha)
    except (OverflowError, ValueError) as e:
        return np.nan
    except Exception as e:
        if "math range error" in str(e).lower():
            return np.nan
        raise e

def calculate_pauc(y_true, y_scores, max_fpr):
    """
    计算局部 ROC-AUC (partial AUC)
    """
    try:
        return roc_auc_score(y_true, y_scores, max_fpr=max_fpr)
    except ValueError:
        return np.nan

# 1. 定义要处理的文件列表
files_info = {
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_compared_pocket_similarity_ligand_efficiency_results_labeled_dep.csv": "compared_pocket_sim",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_keyaas_overlap_ligand_efficiency_results_labeled_dep.csv": "key_aas_smi",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_seq_similarity_ligand_efficiency_results_labeled_dep.csv": "pocket_seq_sim",
    "./MK14/glide-dock_SP_1_MK14_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled_dep.csv": "pocket_sim"
}

# 需要评估的百分比列表
percentages = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
percentage_labels = ["0.1%", "0.2%", "0.5%", "1%", "2%", "5%"]

all_results = []

# 2. 遍历文件并计算所有指标
for file_path, short_name in files_info.items():
    try:
        # 读取数据
        df = pd.read_csv(file_path)

        # ==========================================
        # 新增步骤：先根据 docking score 进行升序排序
        # (对接打分越小越好，因此升序排列把最好的分子排在前面)
        # ==========================================
        df = df.sort_values(by='docking score', ascending=True).reset_index(drop=True)

        y_true = df['Label'].values
        # 翻转 docking score，使其变为越大越好，适配 sklearn 和 rdkit 逻辑
        y_scores = -df['docking score'].values

        row_data = {"Method": short_name}

        for pct, label in zip(percentages, percentage_labels):
            # 计算各项指标
            ef_val = calculate_ef(y_true, y_scores, pct)
            bedroc_val = calculate_bedroc_at_percentage(y_true, y_scores, pct)
            pauc_val = calculate_pauc(y_true, y_scores, pct)

            # 记录到字典中，并处理 NaN 的情况
            row_data[f"EF {label}"] = round(ef_val, 2)
            row_data[f"BEDROC {label}"] = round(bedroc_val, 4) if pd.notna(bedroc_val) else "N/A"
            row_data[f"pAUC {label}"] = round(pauc_val, 4) if pd.notna(pauc_val) else "N/A"

        all_results.append(row_data)

    except Exception as e:
        print(f"处理 {short_name} 时出错: {e}")

# 3. 整理和展示结果
results_df = pd.DataFrame(all_results)

for index, row in results_df.iterrows():
    print(f"\n[{row['Method']}] 早期富集指标:")
    print("-" * 55)
    print(f"{'Top %':<10} | {'EF':<10} | {'BEDROC':<10} | {'pAUC':<10}")
    print("-" * 55)
    for label in percentage_labels:
        print(f"{label:<10} | {str(row[f'EF {label}']):<10} | {str(row[f'BEDROC {label}']):<10} | {str(row[f'pAUC {label}']):<10}")
    print("-" * 55)

In [ ]:
import pandas as pd
import numpy as np
import math
from sklearn.metrics import roc_auc_score
from rdkit.ML.Scoring import Scoring

def calculate_ef(y_true, y_scores, percentage):
    """
    计算特定百分比下的富集因子 (Enrichment Factor, EF)
    """
    n_total = len(y_true)
    n_top = max(1, int(n_total * percentage))
    n_actives = sum(y_true)

    if n_actives == 0:
        return 0.0

    # 此时 y_scores 已经是按从大到小（对接打分从小到大）对齐的
    # 保险起见，依然通过 argsort 获取最高分的索引
    sorted_indices = np.argsort(y_scores)[::-1]
    y_true_sorted = np.array(y_true)[sorted_indices]
    actives_in_top = sum(y_true_sorted[:n_top])

    ef = (actives_in_top / n_top) / (n_actives / n_total)
    return ef

def calculate_bedroc_at_percentage(y_true, y_scores, percentage):
    """
    计算特定百分比下的 BEDROC
    根据公式 alpha = -ln(0.2) / percentage，将 80% 的评价权重集中在前 percentage 的分子上
    """
    alpha = 1.609438 / percentage
    data = list(zip(y_scores, y_true))
    # RDKit 要求输入数据按分数降序排列
    data.sort(key=lambda x: x[0], reverse=True)

    try:
        # 当 alpha 大于 ~709 时，RDKit 底层可能会因为 e^alpha 溢出抛出异常
        return Scoring.CalcBEDROC(data, 1, alpha)
    except (OverflowError, ValueError) as e:
        return np.nan
    except Exception as e:
        if "math range error" in str(e).lower():
            return np.nan
        raise e

def calculate_pauc(y_true, y_scores, max_fpr):
    """
    计算局部 ROC-AUC (partial AUC)
    """
    try:
        return roc_auc_score(y_true, y_scores, max_fpr=max_fpr)
    except ValueError:
        return np.nan

# 1. 定义要处理的文件列表
files_info = {
    "./KIT/glide-dock_SP_1_KIT_all_molecules_by_compared_pocket_similarity_ligand_efficiency_results_labeled_dep.csv": "compared_pocket_sim",
    "./KIT/glide-dock_SP_1_KIT_all_molecules_by_keyaas_overlap_ligand_efficiency_results_labeled_dep.csv": "key_aas_smi",
    "./KIT/glide-dock_SP_1_KIT_all_molecules_by_pocket_seq_similarity_ligand_efficiency_results_labeled_dep.csv": "pocket_seq_sim",
    "./KIT/glide-dock_SP_1_KIT_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled_dep.csv": "pocket_sim"
}

# 需要评估的百分比列表
percentages = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
percentage_labels = ["0.1%", "0.2%", "0.5%", "1%", "2%", "5%"]

all_results = []

# 2. 遍历文件并计算所有指标
for file_path, short_name in files_info.items():
    try:
        # 读取数据
        df = pd.read_csv(file_path)

        # ==========================================
        # 新增步骤：先根据 docking score 进行升序排序
        # (对接打分越小越好，因此升序排列把最好的分子排在前面)
        # ==========================================
        df = df.sort_values(by='docking score', ascending=True).reset_index(drop=True)

        y_true = df['Label'].values
        # 翻转 docking score，使其变为越大越好，适配 sklearn 和 rdkit 逻辑
        y_scores = -df['docking score'].values

        row_data = {"Method": short_name}

        for pct, label in zip(percentages, percentage_labels):
            # 计算各项指标
            ef_val = calculate_ef(y_true, y_scores, pct)
            bedroc_val = calculate_bedroc_at_percentage(y_true, y_scores, pct)
            pauc_val = calculate_pauc(y_true, y_scores, pct)

            # 记录到字典中，并处理 NaN 的情况
            row_data[f"EF {label}"] = round(ef_val, 2)
            row_data[f"BEDROC {label}"] = round(bedroc_val, 4) if pd.notna(bedroc_val) else "N/A"
            row_data[f"pAUC {label}"] = round(pauc_val, 4) if pd.notna(pauc_val) else "N/A"

        all_results.append(row_data)

    except Exception as e:
        print(f"处理 {short_name} 时出错: {e}")

# 3. 整理和展示结果
results_df = pd.DataFrame(all_results)

for index, row in results_df.iterrows():
    print(f"\n[{row['Method']}] 早期富集指标:")
    print("-" * 55)
    print(f"{'Top %':<10} | {'EF':<10} | {'BEDROC':<10} | {'pAUC':<10}")
    print("-" * 55)
    for label in percentage_labels:
        print(f"{label:<10} | {str(row[f'EF {label}']):<10} | {str(row[f'BEDROC {label}']):<10} | {str(row[f'pAUC {label}']):<10}")
    print("-" * 55)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import AllChem, DataStructs
from rdkit.DataStructs import ConvertToNumpyArray

# 设置全局绘图风格，保证图片美观
plt.rcParams['font.sans-serif'] = ['SimHei']  # 解决中文显示
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# ---------------------- 1. 读取原始配体数据 ----------------------
fragments_df = pd.read_csv('./MK14/MK14_fragments_file_by_compared_pocket_similarity_refined_1000_nik.csv')
# 提取第一列的原始配体SMILES，去重
original_smiles = fragments_df.iloc[:, 0].dropna().unique().tolist()
print(f"去重后原始配体数量：{len(original_smiles)}")

# ---------------------- 2. 读取生成分子数据 ----------------------
new_molecules_df = pd.read_csv('./MK14/MK14_new_molecules_file_by_compared_pocket_similarity_refined_filter_10000_nik.csv')
# 提取最后一列的生成分子SMILES
new_smiles = new_molecules_df.iloc[:, -1].dropna().tolist()
print(f"生成分子数量：{len(new_smiles)}")

# ---------------------- 3. 过滤无效SMILES（解析失败的分子） ----------------------
def filter_valid_smiles(smiles_list):
    valid_smiles = []
    for smi in smiles_list:
        try:
            mol = Chem.MolFromSmiles(smi)
            if mol is not None:
                valid_smiles.append(smi)
        except:
            continue
    return valid_smiles

original_smiles = filter_valid_smiles(original_smiles)
new_smiles = filter_valid_smiles(new_smiles)
print(f"有效原始配体数量：{len(original_smiles)}")
print(f"有效生成分子数量：{len(new_smiles)}")

In [ ]:
# 定义ECFP指纹生成函数（半径2=ECFP4，2048位）
def generate_ecfp4_fingerprint(smiles_list):
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        # 生成Morgan指纹（ECFP4）
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
        # 转为numpy数组，方便后续计算
        fp_array = np.zeros((1,), dtype=np.int8)
        ConvertToNumpyArray(fp, fp_array)
        fps.append(fp_array)
    return np.array(fps)

# 生成原始配体和生成分子的指纹
original_fps = generate_ecfp4_fingerprint(original_smiles)
new_fps = generate_ecfp4_fingerprint(new_smiles)
print(f"原始配体指纹矩阵形状：{original_fps.shape}")
print(f"生成分子指纹矩阵形状：{new_fps.shape}")

In [ ]:
# 批量计算Tanimoto相似性，返回形状为[生成分子数量, 原始配体数量]的相似性矩阵
def calculate_tanimoto_matrix(new_fps, original_fps):
    # 计算分子的特征数（指纹中1的数量）
    new_fp_sum = new_fps.sum(axis=1, keepdims=True)  # [n_new, 1]
    original_fp_sum = original_fps.sum(axis=1, keepdims=True)  # [n_original, 1]

    # 计算共同特征数（矩阵点积）
    intersection = np.dot(new_fps, original_fps.T)  # [n_new, n_original]

    # 计算Tanimoto相似性
    union = new_fp_sum + original_fp_sum.T - intersection
    tanimoto_matrix = intersection / union

    return tanimoto_matrix

# 计算相似性矩阵
tanimoto_matrix = calculate_tanimoto_matrix(new_fps, original_fps)
# 每个生成分子取与原始配体的最大相似性
max_similarity = tanimoto_matrix.max(axis=1)
print(f"最大相似性数组形状：{max_similarity.shape}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(12, 7), dpi=300)

# 绘制直方图+核密度曲线
ax = sns.histplot(
    max_similarity,
    bins=50,
    kde=True,
    color='#2B72D2',
    edgecolor='black',
    alpha=0.8,
    linewidth=0.8
)

# 添加均值参考线
mean_val = np.mean(max_similarity)
ax.axvline(mean_val, color='#fdae61', linestyle='--', linewidth=2, label=f'Mean: {mean_val:.3f}')

# 图表设置
ax.set_title('Distribution of Tanimoto Similarity (ECFP4) Between Generated Molecules and Original Ligands', fontsize=16, fontweight='bold')
# X轴标签加粗
ax.set_xlabel('Tanimoto Similarity (ECFP4 Fingerprint)', fontsize=15, fontweight='bold', labelpad=10)
# Y轴标签加粗
ax.set_ylabel('Number of Generated Molecules', fontsize=15, fontweight='bold', labelpad=10)
# 坐标轴刻度数字也加粗
plt.setp(ax.get_xticklabels(), fontweight='bold', fontsize=14)
plt.setp(ax.get_yticklabels(), fontweight='bold', fontsize=14)

# 放大图例：字号+内边距+边框粗细
ax.legend(
    fontsize=16, # 文字放大
    prop={'weight': 'bold', 'size': 11},
    borderpad=1.0,      # 图例内部上下左右留白
    handletextpad=0.8,  # 线条和文字间距
    frameon=True,
    edgecolor='black'
)
ax.set_xlim(0, 1)

# 美化边框
sns.despine()
plt.tight_layout()

# 保存图片
plt.savefig('./MK14/MK14_similarity_distribution_ECFP4_by_compared_pocket_similarity.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image, ImageDraw, ImageFont
import io, math

CSV_PATH = "./MK14/glide-dock_SP_1_MK14_all_molecules_by_compared_pocket_similarity.csv"
N_TOP = 50
FONT_SIZE = 32          # 标签字号(调大: 原默认很小)
MOLS_PER_ROW = 5        # 每行分子数
MOL_SIZE = (450, 450)   # 每个分子结构图尺寸

# 1. 读取并排序(对接分数从低到高, 越小越优)
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]
df_sorted = df.sort_values("docking score", ascending=True).reset_index(drop=True)
top = df_sorted.head(N_TOP).copy()
top["rank"] = range(1, N_TOP + 1)

# 2. SMILES -> Mol
top["mol"] = top["Original_SMILES"].apply(lambda s: Chem.MolFromSmiles(s))
top = top[top["mol"].notna()].reset_index(drop=True)

# 3. 画网格(每格: 分子结构 + 加粗居中的 "#rank docking score" 标签)
def draw_grid(mols, scores, ranks, per_row=MOLS_PER_ROW, size=MOL_SIZE, font_size=FONT_SIZE):
    rows = math.ceil(len(mols) / per_row)
    cw, ch = size[0] + 40, size[1] + 90   # 底部多留空间给更大的标签
    canvas = Image.new("RGB", (cw * per_row, ch * rows), "white")
    draw = ImageDraw.Draw(canvas)

    # 加粗字体(优先粗体 TTF, 找不到则回退默认)
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", font_size)
    except Exception:
        try:
            font = ImageFont.truetype("arialbd.ttf", font_size)
        except Exception:
            font = ImageFont.load_default()

    for i, (m, s, r) in enumerate(zip(mols, scores, ranks)):
        d2d = rdMolDraw2D.MolDraw2DCairo(*size)
        rdMolDraw2D.PrepareAndDrawMolecule(d2d, m)
        d2d.FinishDrawing()
        img = Image.open(io.BytesIO(d2d.GetDrawingText())).convert("RGB")
        x, y = (i % per_row) * cw, (i // per_row) * ch
        canvas.paste(img, (x + (cw - size[0]) // 2, y + 20))

        label = f"{r}  docking score {s:.2f}"
        # 计算文本宽度, 在单元格内水平居中
        bbox = draw.textbbox((0, 0), label, font=font)
        tw = bbox[2] - bbox[0]
        label_x = x + (cw - tw) // 2
        label_y = y + size[1] + 30
        draw.text((label_x, label_y), label, fill="black", font=font)
    return canvas

grid = draw_grid(top["mol"], top["docking score"], top["rank"])
grid.save("./MK14/top50_docking_structures_compared_pocket_similarity_SP.png")
print("已保存:")


In [ ]:
import pandas as pd
from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D
from PIL import Image, ImageDraw, ImageFont
import io, math

# 配置参数
CSV_PATH = "./KIT/glide-dock_SP_1_KIT_all_molecules_by_pocket_similarity_ligand_efficiency_results_labeled.csv"
N_TOP = 50  # 选取前50个分子
LABEL_FILTER = 0  # 筛选Label为0的分子
MOLS_PER_ROW = 5  # 每行展示的分子数
FONT_SIZE = 32  # 标签字号
MOL_SIZE = (450, 450)  # 单个分子图的尺寸

# 1. 读取并预处理数据
df = pd.read_csv(CSV_PATH)
df.columns = [c.strip() for c in df.columns]  # 去除列名前后空格

# 校验必需列
required_cols = {"Label", "docking score", "Original_SMILES"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV 缺少必需列: {missing}")

# 2. 筛选Label=0的分子，按对接分数从低到高排序（越小越优）
df_filtered = df[df["Label"] == LABEL_FILTER].copy()
df_sorted = df_filtered.sort_values("docking score", ascending=True).reset_index(drop=True)

# 3. 取前N_TOP个分子，添加排名
top = df_sorted.head(N_TOP).copy()
top["rank"] = range(1, len(top) + 1)

# 4. SMILES 转 Mol 对象，过滤无效结构
top["mol"] = top["Original_SMILES"].apply(lambda s: Chem.MolFromSmiles(s) if pd.notna(s) else None)
top = top[top["mol"].notna()].reset_index(drop=True)

if top.empty:
    raise ValueError("筛选后无有效分子（SMILES 全部解析失败）")

print(f"筛选后Label=0的分子总数：{len(df_filtered)}")
print(f"成功解析的前{len(top)}个分子，对接分数范围：{top['docking score'].min():.2f} ~ {top['docking score'].max():.2f}")

# 5. 绘制分子结构网格图（标签调大、加粗、居中）
def draw_grid(mols, scores, ranks, per_row=MOLS_PER_ROW, size=MOL_SIZE, font_size=FONT_SIZE):
    rows = math.ceil(len(mols) / per_row)
    # 单元格宽高：分子尺寸 + 边距 + 标签空间
    cell_w, cell_h = size[0] + 40, size[1] + 90
    canvas = Image.new("RGB", (cell_w * per_row, cell_h * rows), "white")
    draw = ImageDraw.Draw(canvas)

    # 加载加粗字体，兼容不同系统
    try:
        font = ImageFont.truetype("DejaVuSans-Bold.ttf", font_size)
    except:
        try:
            font = ImageFont.truetype("arialbd.ttf", font_size)
        except:
            font = ImageFont.load_default(size=font_size)

    for i, (m, s, r) in enumerate(zip(mols, scores, ranks)):
        # 绘制分子结构
        d2d = rdMolDraw2D.MolDraw2DCairo(*size)
        rdMolDraw2D.PrepareAndDrawMolecule(d2d, m)
        d2d.FinishDrawing()
        img = Image.open(io.BytesIO(d2d.GetDrawingText())).convert("RGB")

        # 计算分子在单元格内的居中位置
        x, y = (i % per_row) * cell_w, (i // per_row) * cell_h
        mol_x = x + (cell_w - size[0]) // 2
        mol_y = y + 20
        canvas.paste(img, (mol_x, mol_y))

        # 绘制标签：加粗、居中
        label = f"#{r}  docking score {s:.2f}"
        # 计算文本宽度实现居中
        text_bbox = draw.textbbox((0, 0), label, font=font)
        text_width = text_bbox[2] - text_bbox[0]
        text_x = x + (cell_w - text_width) // 2
        text_y = y + size[1] + 30
        draw.text((text_x, text_y), label, font=font, fill="black")

    return canvas

# 生成并保存图片
grid_img = draw_grid(top["mol"], top["docking score"], top["rank"])
output_path = "./KIT/top50_docking_structures_pocket_similarity_SP.png"
grid_img.save(output_path)

# 同时保存明细表，方便后续查看
top_detail = top[["rank", "Title", "Original_SMILES", "docking score", "Heavy_Atoms", "Ligand_Efficiency"]]
# top_detail.to_csv("top50_label0_docking_detail.csv", index=False)

print(f"图片已保存至：{output_path}")
print(f"明细表已保存至：top50_label0_docking_detail.csv")

# 展示图片
grid_img.show()